# 05 - The user x genre table

**Goal:** for every user and every year, count how many listens they had in each genre. This is the table that looks at the data from the **listeners' side** (notebooks 02-03 looked from the genres' side). Part 2 (which genres share an audience) and the diversity questions start from here.

**Long format.** The table has one row per `(year, user_id, genre)`, not one column per genre. A user only gets rows for the genres they actually listened to. A wide matrix (one column per genre) is made later when an analysis needs it, from this table, as a sparse matrix.

| column | meaning |
|---|---|
| `year`, `user_id`, `genre` | the key |
| `listens` | the user's listens in this genre in this year |
| `share` | `listens` divided by the user's labelled listens that year (the shares of one user-year add up to 1) |

A second small table has one row per `(year, user_id)` with `total_listens` (all listens that year), `labelled_listens` (the ones that have a genre, the denominator of `share`), `coverage_pct` (labelled / total, in percent) and `bucket` (0-9 = development, 10-19 = validation).

Both tables are saved twice: as parquet in `Output_dev/data/parquets/user_genre_tables/` and as csv in `Output_dev/user_genre_tables/`.

How it is made, for each year:
1. take the user's listens from the extract, without the duplicates flagged by `is_dup` (as in notebook 02),
2. look up the genre of each listen in the song table that notebook 02 saved (`songs_genre_YEAR_...parquet`, joined on artist + title),
3. count listens per user and genre. Listens without a genre are counted in the user's total, but get no row in the genre table.

## 1. Setup

In [ ]:
%pip install -q duckdb pyarrow

In [ ]:
from pathlib import Path
import os, time
import duckdb
import pandas as pd

RAW = Path("/Users/hanshornbek/Desktop/Projects/Data Mining project")
EXTRACT = RAW / "Extract"

DEV_ONLY = os.environ.get("DEV_ONLY", "1") == "1"      # True: only the development users (bucket 0-9)
YEARS = range(2006, 2027)                              # all the years we have song tables for
MEMORY_LIMIT = "3GB"

SUFFIX = "dev" if DEV_ONLY else "all"
OUT = Path.cwd().parent / "Output_dev"
SONGS = OUT / "data" / "parquets" / "songs"            # the song tables from notebook 02
PQ = OUT / "data" / "parquets" / "user_genre_tables"   # the two tables as parquet
CSV = OUT / "user_genre_tables"                        # the two tables as csv
PQ.mkdir(parents=True, exist_ok=True)
CSV.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"PRAGMA memory_limit='{MEMORY_LIMIT}'").fetchall()
con.execute(f"PRAGMA temp_directory='{OUT / 'data' / '.tmp'}'").fetchall()

users = pd.read_csv(EXTRACT / "cohort_users.csv")
keep = users[users["bucket"] < 10] if DEV_ONLY else users
con.register("keep_users", keep[["user_id"]])
print(f"{len(keep):,} users ({'development users only' if DEV_ONLY else 'all users'}), years {YEARS[0]}-{YEARS[-1]}")

## 2. Count listens per user and genre

The query joins every listen to its song in the song table (`LEFT JOIN`, so listens whose song has no genre stay in). `GROUP BY user_id, genre` then counts. A missing genre is `NULL`; these rows are used for the user's total and then left out of the genre table.

In [ ]:
long_parts, total_parts = [], []

for year in YEARS:
    start = time.time()
    files = EXTRACT / "extract" / f"year={year}" / "*.parquet"
    songs = SONGS / f"songs_genre_{year}_{SUFFIX}.parquet"
    counts = con.sql(f"""
        WITH l AS (
            SELECT user_id, coalesce(artist_name, '') AS artist_name, coalesce(track_name, '') AS track_name
            FROM read_parquet('{files}')
            WHERE user_id IN (SELECT user_id FROM keep_users) AND NOT coalesce(is_dup, false)
        ), s AS (SELECT artist_name, track_name, genre FROM read_parquet('{songs}'))
        SELECT {year} AS year, l.user_id, s.genre, count(*) AS listens
        FROM l LEFT JOIN s USING (artist_name, track_name)
        GROUP BY 1, 2, 3
    """).df()

    # per user and year: all listens, and the listens that have a genre
    totals = counts.groupby(["year", "user_id"])["listens"].sum().rename("total_listens").reset_index()
    labelled = counts[counts["genre"].notna()].groupby(["year", "user_id"])["listens"].sum().rename("labelled_listens").reset_index()
    totals = totals.merge(labelled, on=["year", "user_id"], how="left").fillna({"labelled_listens": 0})
    long_parts.append(counts[counts["genre"].notna()])
    total_parts.append(totals)

    # CHECK 1: all listens = what the song table says for this year
    song_total = con.execute(f"SELECT sum(n_listens) FROM read_parquet('{songs}')").fetchall()[0][0]
    assert totals["total_listens"].sum() == song_total, f"{year}: totals differ from the song table"
    print(f"{year}: {len(totals):6,} users, {len(long_parts[-1]):8,} rows, {totals['total_listens'].sum():11,} listens ({time.time() - start:.0f} s)")

## 3. Check against notebook 02, and save

The genre listens of all users together must equal the `listens` in notebook 02's genre counts, genre by genre. If one number differs, the notebook stops.

In [ ]:
user_genre = pd.concat(long_parts, ignore_index=True)
user_year = pd.concat(total_parts, ignore_index=True)
user_year = user_year.merge(users[["user_id", "bucket"]], on="user_id", how="left")
user_year["labelled_listens"] = user_year["labelled_listens"].astype("int64")

# share = the user's listens in this genre / all the user's labelled listens that year
user_genre = user_genre.merge(user_year[["year", "user_id", "labelled_listens"]], on=["year", "user_id"], how="left")
user_genre["share"] = user_genre["listens"] / user_genre["labelled_listens"]
user_genre = user_genre.drop(columns="labelled_listens")

# CHECK 2: summed over users, our table must give the same numbers as genre_counts from notebook 02
ours = user_genre.groupby(["year", "genre"])["listens"].sum()
theirs = pd.concat([pd.read_csv(OUT / "genre_counts" / str(y) / f"genre_counts_{y}_{SUFFIX}.csv") for y in YEARS]).set_index(["year", "genre"])["listens"]
diff = ours.subtract(theirs, fill_value=0)
assert (diff == 0).all(), f"{(diff != 0).sum()} year-genre combinations differ from notebook 02"
print("the same as the genre counts from notebook 02, for every year and genre")

# CHECK 3: labelled listens = the sum of the genre rows, per user-year
rows_sum = user_genre.groupby(["year", "user_id"])["listens"].sum()
check = user_year.set_index(["year", "user_id"])["labelled_listens"]
check = check[check > 0]
assert (rows_sum == check.reindex(rows_sum.index)).all() and len(rows_sum) == len(check), "labelled listens differ from the genre rows"

# CHECK 4: the shares of every user-year add up to 1
share_sum = user_genre.groupby(["year", "user_id"])["share"].sum()
assert ((share_sum - 1).abs() < 1e-9).all(), "shares do not add up to 1"
print("shares add up to 1 for every user-year")

# the user-years count should be the same as the extract's own table (users with listens that year, after dropping duplicates)
print(f"{len(user_year):,} user-years, {len(user_genre):,} rows (year, user, genre), {user_genre['genre'].nunique():,} different genres")

user_genre["listens"] = user_genre["listens"].astype("int32")
user_genre = user_genre[["year", "user_id", "genre", "listens", "share"]]
user_year["coverage_pct"] = (100 * user_year["labelled_listens"] / user_year["total_listens"]).round(1)     # % of the user's listens that has a genre
user_year = user_year[["year", "user_id", "total_listens", "labelled_listens", "coverage_pct", "bucket"]]

# sort: by year and user (in user_genre the biggest genre of a user comes first)
user_genre = user_genre.sort_values(["year", "user_id", "share"], ascending=[True, True, False], ignore_index=True)
user_year = user_year.sort_values(["year", "user_id"], ignore_index=True)
for name, table in [("user_genre", user_genre), ("user_year", user_year)]:
    table.to_parquet(PQ / f"{name}_{SUFFIX}.parquet", index=False)
    table.to_csv(CSV / f"{name}_{SUFFIX}.csv", index=False)
print("saved parquet to", PQ)
print("saved csv to", CSV)

## 4. A first look

How many genres does a typical user have in a year, and how much of their listening has a genre?

In [ ]:
per = user_genre.groupby(["year", "user_id"]).size().rename("n_genres").reset_index().merge(user_year, on=["year", "user_id"])
per["coverage"] = per["coverage_pct"] / 100
per.groupby("year")[["n_genres", "total_listens", "coverage"]].median().round(2)